In [9]:
import pandas as pd

In [5]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "automobile_dataset.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "ranaghulamnabi/automobile-market-dataset-vehicle-specification",
    file_path,
)

df.head()

C:\Users\idowe\AppData\Local\Temp\ipykernel_21076\1447675060.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


,Make,Model,Year,Fuel_Type,Transmission,Engine_Size,Mileage,Horsepower,Torque,Owners,Accident_History,Service_History,Color,Body_Type,Drivetrain,Fuel_Efficiency,Location,Selling_Price
0,Mercedes-Benz,GLE,2024,Petrol,Automatic,2.3,100,186.0,196.0,1,0.0,NaN,Silver,SUV,AWD,30.0,IL,64140
1,Hyundai,Tucson,2008,Petrol,Automatic,2.3,387035,189.0,190.0,4,0.0,NaN,Blue,SUV,FWD,35.0,FL,500
2,Volkswagen,Golf,2021,Hybrid,NaN,1.9,46054,158.0,153.0,1,0.0,Partial Service,Gray,Hatchback,AWD,43.0,NY,16429
3,Chevrolet,Tahoe,2005,Petrol,NaN,2.2,141302,169.0,165.0,5,1.0,No Service,Blue,SUV,AWD,NaN,FL,2199
4,Toyota,Camry,2022,Petrol,Automatic,1.9,32813,149.0,141.0,1,0.0,Full Service,Brown,Sedan,FWD,38.0,CA,21792


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5500 entries, 0 to 5499
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Make              5500 non-null   str    
 1   Model             5500 non-null   str    
 2   Year              5500 non-null   int64  
 3   Fuel_Type         5500 non-null   str    
 4   Transmission      4673 non-null   str    
 5   Engine_Size       4656 non-null   float64
 6   Mileage           5500 non-null   int64  
 7   Horsepower        4700 non-null   float64
 8   Torque            4701 non-null   float64
 9   Owners            5500 non-null   int64  
 10  Accident_History  4688 non-null   float64
 11  Service_History   4645 non-null   str    
 12  Color             4697 non-null   str    
 13  Body_Type         5500 non-null   str    
 14  Drivetrain        5500 non-null   str    
 15  Fuel_Efficiency   4706 non-null   float64
 16  Location          4718 non-null   str    
 17  Sellin

In [7]:
df.isnull().sum()

Make                  0
Model                 0
Year                  0
Fuel_Type             0
Transmission        827
Engine_Size         844
Mileage               0
Horsepower          800
Torque              799
Owners                0
Accident_History    812
Service_History     855
Color               803
Body_Type             0
Drivetrain            0
Fuel_Efficiency     794
Location            782
Selling_Price         0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [13]:
# Extract all column names from the DataFrame as a list
all_cols = list(df.columns)

print("--- Sequential Duplicates Analysis ---")

# Variables to track duplicate counts and record discriminating power of dropped columns
prev_dup_count = 0
max_delta = 0
max_delta_col = None

# List to store all discriminating columns sorted by their impact (delta)
discriminating_cols = []

# LOGIC SUMMARY:
# Iterate backwards from all columns down to 1 to measure sensitivity to feature removal.
# A sudden spike in duplicate count (high Delta) indicates that the dropped column 
# possessed strong discriminating power (variance) required to keep rows unique.
for i in range(len(all_cols), 0, -1):
    # Select the first 'i' columns from the list as the evaluation subset
    subset = all_cols[:i]
    
    # Count the number of duplicate rows based only on the current subset of columns
    dup_count = df.duplicated(subset=subset).sum()
    
    # Calculate the gap (delta) from the previous step (skip calculation for the full feature set)
    if i < len(all_cols):
        delta = dup_count - prev_dup_count
        dropped_col = all_cols[i]
        
        # LOGIC: Track the maximum jump in duplicates to pinpoint the core discriminator column
        if delta > max_delta:
            max_delta = delta
            max_delta_col = dropped_col
            
        # Record the dropped column and its impact if it created new duplicates
        if delta > 0:
            discriminating_cols.append({'column': dropped_col, 'delta': delta})
            
        print(f"Based on first {i} columns: {dup_count} duplicates (Δ = +{delta})")
    else:
        print(f"Based on first {i} columns: {dup_count} duplicates")
        
    # Update previous duplicate count for the next iteration
    prev_dup_count = dup_count

# Sort discriminating columns by impact in descending order
discriminating_cols.sort(key=lambda x: x['delta'], reverse=True)

print("-" * 50)
print(f"Largest Duplicate Jump: +{max_delta} duplicates")
print(f"Critical Column Dropped: '{max_delta_col}'")

print("\n--- Ordered Discriminating Columns (Ranked by Impact) ---")
for rank, item in enumerate(discriminating_cols, start=1):
    print(f"{rank}. Column '{item['column']}' -> Added +{item['delta']} duplicates when dropped")

--- Sequential Duplicates Analysis ---
Based on first 18 columns: 0 duplicates
Based on first 17 columns: 0 duplicates (Δ = +0)
Based on first 16 columns: 0 duplicates (Δ = +0)
Based on first 15 columns: 0 duplicates (Δ = +0)
Based on first 14 columns: 0 duplicates (Δ = +0)
Based on first 13 columns: 0 duplicates (Δ = +0)
Based on first 12 columns: 0 duplicates (Δ = +0)
Based on first 11 columns: 0 duplicates (Δ = +0)
Based on first 10 columns: 0 duplicates (Δ = +0)
Based on first 9 columns: 0 duplicates (Δ = +0)
Based on first 8 columns: 0 duplicates (Δ = +0)
Based on first 7 columns: 7 duplicates (Δ = +7)
Based on first 6 columns: 485 duplicates (Δ = +478)
Based on first 5 columns: 3065 duplicates (Δ = +2580)
Based on first 4 columns: 3816 duplicates (Δ = +751)
Based on first 3 columns: 4701 duplicates (Δ = +885)
Based on first 2 columns: 5460 duplicates (Δ = +759)
Based on first 1 columns: 5490 duplicates (Δ = +30)
--------------------------------------------------
Largest Duplicate

In [ ]:
kagglehub.dataset_purge() # delete the dataset from the local path